# Notebook 08: Pydantic Output Parser

In this notebook, we will learn how to **structure the LLM's output** using a Pydantic Output Parser.

## What You Will Learn

- What Pydantic is and why we use it
- How to create structured output schemas
- How to parse LLM responses into Python objects
- How to return structured data (Answer, Source Page, Confidence, Context)

## What is Pydantic?

**Pydantic** is a Python library for data validation. We use it to define the **exact structure** we want the LLM to return.

Without a parser, the LLM returns raw text:
```
"The document discusses AI. It was written in 2024."
```

With a Pydantic parser, the LLM returns structured data:
```python
{
    "answer": "The document discusses AI.",
    "source_page": 5,
    "confidence": "high",
    "retrieved_context": "AI is transforming..."
}
```

## Why Do We Need Structured Output?

Our Gradio GUI (Notebook 09) needs to display:
- The answer text
- The source page number
- A confidence level
- The context used

Raw text from the LLM can't be split into these fields automatically.

## Step 1: Import Required Libraries

In [1]:
from pydantic import BaseModel, Field
from typing import Literal
from langchain_ollama import OllamaLLM
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.runnables import RunnablePassthrough

print("All imports successful!")

C:\Users\Ahmed\AppData\Local\Temp\ipykernel_2868\1213901118.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


All imports successful!


## Step 2: Define the Output Schema

We create a Pydantic model that defines the exact fields we want from the LLM.

In [2]:
class RAGResponse(BaseModel):
    """
    Structured output schema for RAG responses.
    
    Fields:
    - answer: The answer to the user's question
    - source_page: The page number where the answer was found
    - confidence: How confident we are (high/medium/low/not_found)
    - retrieved_context: The context used to answer the question
    """
    answer: str = Field(description="The answer to the user's question")
    source_page: str = Field(description="The page number where the answer was found")
    confidence: Literal["high", "medium", "low", "not_found"] = Field(
        description="Confidence level of the answer"
    )
    retrieved_context: str = Field(description="The retrieved context used to generate the answer")

print("RAGResponse schema created!")
print(f"Fields: {list(RAGResponse.model_fields.keys())}")

RAGResponse schema created!
Fields: ['answer', 'source_page', 'confidence', 'retrieved_context']


## Step 3: Create the Output Parser

In [3]:
# Create a Pydantic output parser
parser = PydanticOutputParser(pydantic_object=RAGResponse)

# The parser generates format instructions for the prompt
format_instructions = parser.get_format_instructions()
print("Format instructions:")
print(format_instructions)

Format instructions:
The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"description": "Structured output schema for RAG responses.\n\nFields:\n- answer: The answer to the user's question\n- source_page: The page number where the answer was found\n- confidence: How confident we are (high/medium/low/not_found)\n- retrieved_context: The context used to answer the question", "properties": {"answer": {"description": "The answer to the user's question", "title": "Answer", "type": "string"}, "source_page": {"description": "The page number where the answer was found", "title": "Source Page", "type":

## Step 4: Setup the RAG Pipeline

In [4]:
# Setup (reusing previous code)
loader = PyPDFLoader("../data/sample.pdf")
pages = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = text_splitter.split_documents(pages)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

llm = OllamaLLM(model="mistral", temperature=0.0, num_ctx=4096)

def format_docs(docs):
    return "\n\n".join([
        f"[{i+1}] (Page {doc.metadata.get('page', '?')})\n{doc.page_content}"
        for i, doc in enumerate(docs)
    ])

print("Setup complete!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Setup complete!


## Step 5: Create the Structured Prompt

In [5]:
# Create a prompt that includes the format instructions
rag_prompt = PromptTemplate.from_template("""
You are a precise document analyst. Answer questions using ONLY the provided context.

## Document Context:
{context}

## Question:
{question}

## Strict Rules:
1. Answer using ONLY the provided context.
2. If the answer is not in the context, set confidence to "not_found" and explain what's missing.
3. Do not hallucinate or use outside knowledge.

{format_instructions}
""").partial(format_instructions=format_instructions)

print("Structured prompt created!")

Structured prompt created!


## Step 6: Build the Chain with Output Parsing

In [6]:
# Build the RAG chain with output parsing
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | parser
)

print("RAG chain with output parser built!")

RAG chain with output parser built!


## Step 7: Test the Structured Output

In [7]:
# Test with a question that should be answerable
question = "What is the main topic of this document?"

try:
    result = rag_chain.invoke(question)
    
    print(f"Question: {question}")
    print(f"\nParsed Response:")
    print(f"  Answer: {result.answer}")
    print(f"  Source Page: {result.source_page}")
    print(f"  Confidence: {result.confidence}")
    print(f"  Context: {result.retrieved_context[:100]}...")
except Exception as e:
    print(f"Error: {e}")
    print("\nNote: Ollama's Mistral may not always produce perfectly formatted output.")
    print("We'll add error handling for this in the final project.")

Question: What is the main topic of this document?

Parsed Response:
  Answer: Artificial Intelligence
  Source Page: Not applicable (Introduction)
  Confidence: high
  Context: Introduction to Artificial Intelligence...


## Step 8: Test with an Unanswerable Question

In [8]:
# Test with a question that should NOT be in the document
question = "What is the capital of France?"

try:
    result = rag_chain.invoke(question)
    
    print(f"Question: {question}")
    print(f"\nParsed Response:")
    print(f"  Answer: {result.answer}")
    print(f"  Source Page: {result.source_page}")
    print(f"  Confidence: {result.confidence}")
    print(f"  Context: {result.retrieved_context[:100]}...")
except Exception as e:
    print(f"Error: {e}")
    print("This is expected - the model may not always follow the format.")

Question: What is the capital of France?

Parsed Response:
  Answer: Not found
  Source Page: Not applicable
  Confidence: not_found
  Context: The provided context does not contain information about the capital of France....


## Step 9: Handle Parsing Errors Gracefully

Sometimes the LLM doesn't follow the format. We need error handling.

In [9]:
def safe_invoke(chain, question):
    """
    Safely invoke the RAG chain with error handling.
    Returns a fallback response if parsing fails.
    """
    try:
        result = chain.invoke(question)
        return result
    except Exception as e:
        print(f"Parsing failed: {e}")
        # Return a fallback response
        return RAGResponse(
            answer="I'm sorry, but I could not parse the response properly.",
            source_page="unknown",
            confidence="low",
            retrieved_context="Error occurred during response parsing."
        )

# Test the safe invoke
result = safe_invoke(rag_chain, "What is the main topic?")
print(f"Safe invoke result: {result}")
print(f"  Answer: {result.answer}")
print(f"  Confidence: {result.confidence}")

Safe invoke result: answer='The main topic is Machine Learning and Artificial Intelligence.' source_page='[2, 3]' confidence='high' retrieved_context='structures. Examples include customer segmentation and anomaly detection.\n3. Reinforcement Learning: The algorithm learns by interacting with an environment and receiving rewards or penalties for its actions. Examples include game playing and robotics.\nPage 1\n\n[2] (Page 0)\nstatistical techniques to find patterns in data.\nThere are three main types of Machine Learning:\n1. Supervised Learning: The algorithm learns from labeled data, meaning each training example is paired with the correct output. Examples include image classification and spam detection.\n2. Unsupervised Learning: The algorithm learns from unlabeled data and tries to find hidden patterns or structures. Examples include customer segmentation and anomaly detection.\n\n[3] (Page 2)\nIntroduction to Artificial Intelligence\nChapter 4: Natural Language Processing'
  Answe

## Key Takeaways

1. **Pydantic** defines the exact structure of the LLM's output
2. **PydanticOutputParser** converts raw LLM text into structured Python objects
3. **Format instructions** in the prompt tell the LLM how to format its response
4. **Error handling** is crucial because LLMs may not always follow the format
5. Our structured output includes: Answer, Source Page, Confidence, Retrieved Context

## Output Schema Summary

```python
class RAGResponse(BaseModel):
    answer: str
    source_page: str
    confidence: "high" | "medium" | "low" | "not_found"
    retrieved_context: str
```

## Next Steps

Proceed to **Notebook 09: Gradio GUI** to build a simple interface!